# 02 - CGCS Prototype for Trisomy 21

**Main Working Prototype**  
**Allen Lab — Residue Manifold Learning**

This notebook tests a simple CGCS prototype for chromosome 21 dosage perturbation.

The working path is:

```text
dosage ratio
→ CGCS components
→ coherence score
→ intervention recovery
```


In [ ]:
# ================================================
# SETUP
# ================================================
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

if (cwd / "src").exists():
    repo_root = cwd
elif cwd.name in {"grok", "chatgpt"} and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]
elif (cwd / REPO_NAME).exists():
    repo_root = cwd / REPO_NAME
else:
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

figures_dir = repo_root / "figures" / "grok"
results_dir = repo_root / "results" / "grok"
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

try:
    import grok
    from grok import *
    from grok.cgcs import calculate_cgcs, calculate_cgcs_trisomy
    from grok.trisomy_metrics import trisomy_cgcs_score, simulate_intervention_recovery
    USING_SRC = True
except Exception:
    USING_SRC = False

    def calculate_cgcs(*components):
        clipped = [max(0.0, min(1.0, float(c))) for c in components]
        product = 1.0
        for c in clipped:
            product *= c
        return product ** (1.0 / len(clipped)) if clipped else 0.0

    def trisomy_cgcs_score(
        dosage_ratio=1.5,
        overexpression_imbalance=0.5,
        global_dysregulation=0.4,
        redundancy_factor=1.0,
        return_components=False,
    ):
        dosage_noise = abs(float(dosage_ratio) - 1.0)
        redundancy_factor = max(float(redundancy_factor), 1e-9)

        components = {
            "dosage_alignment": max(0.0, 1.0 - dosage_noise / redundancy_factor),
            "overexpression_penalty": max(0.0, 1.0 - float(overexpression_imbalance) * dosage_noise),
            "dysregulation_penalty": max(0.0, 1.0 - float(global_dysregulation) * dosage_noise),
            "propagation_alignment": max(0.0, 1.0 - 0.5 * dosage_noise),
        }

        cgcs = calculate_cgcs(*components.values())

        result = {
            "dosage_ratio": float(dosage_ratio),
            "dosage_noise": dosage_noise,
            **components,
            "cgcs": cgcs,
        }

        return result if return_components else {"cgcs": cgcs}

    def simulate_intervention_recovery(baseline_cgcs, recovery_strength=0.45):
        baseline_cgcs = float(baseline_cgcs)
        recovery_strength = max(0.0, min(1.0, float(recovery_strength)))
        return baseline_cgcs + recovery_strength * (1.0 - baseline_cgcs)

print("Setup complete")
print("Repo root:", repo_root)
print("Using src package:", USING_SRC)


## 1. Dosage Sensitivity

The first figure shows how CGCS changes as dosage ratio increases from `1.0x` to `2.0x`.


In [ ]:
dosage_ratios = np.linspace(1.0, 2.0, 21)
cgcs_values = [
    trisomy_cgcs_score(dosage_ratio=r, return_components=True)["cgcs"]
    for r in dosage_ratios
]

dosage_df = pd.DataFrame({
    "dosage_ratio": dosage_ratios,
    "cgcs": cgcs_values,
})

display(dosage_df.head())
display(dosage_df.tail())

plt.figure(figsize=(9, 5))
plt.plot(dosage_ratios, cgcs_values, "o-", linewidth=2.5)
plt.axvline(1.0, linestyle="--", linewidth=1, label="normal 1.0x")
plt.axvline(1.5, linestyle="--", linewidth=1, label="trisomy 1.5x")
plt.title("CGCS Degradation with Increasing Gene Dosage")
plt.xlabel("Dosage ratio")
plt.ylabel("CGCS score")
plt.legend()
plt.grid(True, alpha=0.3)

fig_path = figures_dir / "trisomy21_dosage_sensitivity.png"
plt.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)


## 2. Component Breakdown at 1.5x Dosage

This cell inspects the prototype components for the Trisomy 21 dosage ratio.


In [ ]:
result = trisomy_cgcs_score(
    dosage_ratio=1.5,
    overexpression_imbalance=0.5,
    global_dysregulation=0.4,
    redundancy_factor=1.0,
    return_components=True,
)

component_df = pd.DataFrame(
    [{"component": key, "value": value} for key, value in result.items()]
)

display(component_df)

print(f"CGCS at 1.5x dosage: {result['cgcs']:.4f}")


## 3. Intervention Recovery

The recovery curve models intervention strength as a projection from the lower CGCS baseline toward higher coherence.


In [ ]:
baseline = result["cgcs"]
recovery_strengths = np.linspace(0.0, 0.8, 9)
recovered = [
    simulate_intervention_recovery(baseline, recovery_strength=strength)
    for strength in recovery_strengths
]

recovery_df = pd.DataFrame({
    "recovery_strength": recovery_strengths,
    "recovery_strength_percent": recovery_strengths * 100,
    "recovered_cgcs": recovered,
})

display(recovery_df)

plt.figure(figsize=(9, 5))
plt.plot(recovery_strengths * 100, recovered, "o-", linewidth=2.5, label="recovered CGCS")
plt.axhline(y=baseline, linestyle="--", label="baseline at 1.5x")
plt.title("CGCS Improvement Through Intervention Strength")
plt.xlabel("Intervention strength (%)")
plt.ylabel("CGCS score")
plt.legend()
plt.grid(True, alpha=0.3)

fig_path = figures_dir / "trisomy21_intervention_recovery.png"
plt.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)


## 4. Scenario Comparison

The table compares several simple dosage and dysregulation scenarios.


In [ ]:
scenarios = [
    {"name": "Normal", "ratio": 1.0, "imbalance": 0.0, "dysregulation": 0.0},
    {"name": "Typical Trisomy 21", "ratio": 1.5, "imbalance": 0.5, "dysregulation": 0.35},
    {"name": "High Dysregulation", "ratio": 1.5, "imbalance": 0.8, "dysregulation": 0.6},
    {"name": "Mild Mosaic", "ratio": 1.3, "imbalance": 0.25, "dysregulation": 0.2},
]

rows = []
for scenario in scenarios:
    score_record = trisomy_cgcs_score(
        dosage_ratio=scenario["ratio"],
        overexpression_imbalance=scenario["imbalance"],
        global_dysregulation=scenario["dysregulation"],
        return_components=True,
    )
    rows.append({**scenario, "cgcs": score_record["cgcs"]})

scenario_df = pd.DataFrame(rows)
display(scenario_df)

results_path = results_dir / "trisomy21_cgcs_scenarios.csv"
scenario_df.to_csv(results_path, index=False)
print("Saved:", results_path)


## 5. Key Points

### RML / CGCS key points

- RML studies structured variation on constrained manifolds.
- CGCS scores coherence under constraints.
- Perturbations lower coherence when structure becomes less stable.
- Recovery operations model movement back toward higher coherence.

### Trisomy 21 key points

- Trisomy 21 can be represented as a dosage-ratio perturbation.
- Normal dosage is modeled as `1.0x`.
- Trisomy dosage is modeled as `1.5x`.
- Dosage perturbation lowers the CGCS score in this prototype.

### Combined key points

- RML provides the structural frame.
- CGCS provides the measurable score.
- Trisomy 21 provides the domain-specific perturbation.
- Intervention recovery can be modeled as a projection toward restored coherence.
